# Build Your First MCP Server

We'll use `MCPServer` from the official Python SDK. A tool is just a Python function with a docstring and type hints.


In [ ]:
# !pip install -q "mcp>=2"

In [2]:
%%writefile server_basic.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@mcp.tool()
def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())


if __name__ == "__main__":
    mcp.run()  # stdio transport by default

Overwriting server_basic.py


That's the whole server. The **type hints** become the input schema and the **docstring** becomes the tool description the model reads.

Now let's test it with a tiny client.


In [3]:
import os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def server_params(script):
    return StdioServerParameters(command=sys.executable, args=[script])


# Jupyter's stderr has no real file descriptor, so we send server logs to devnull
DEVNULL = open(os.devnull, "w")

In [4]:
async with stdio_client(server_params("server_basic.py"), errlog=DEVNULL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        for t in tools.tools:
            print(f"{t.name}: {t.description}")

        result = await session.call_tool("add", {"a": 2, "b": 3})
        print("add(2, 3) ->", result.content[0].text)

        result = await session.call_tool("word_count", {"text": "MCP is pretty simple"})
        print("word_count ->", result.content[0].text)

add: Add two numbers.
word_count: Count the words in a piece of text.
add(2, 3) -> 5
word_count -> 4


## Bonus: the MCP Inspector

In a terminal you can open a visual debugger:

```
npx @modelcontextprotocol/inspector python server_basic.py
```

## Recap

- `MCPServer("name")` creates the server
- `@mcp.tool()` turns a function into a tool
- `mcp.run()` starts it over stdio

**Up next:** resources and prompts.
